<a href="https://colab.research.google.com/github/cpython-projects/python_da_17_03_26/blob/main/lesson_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔷 ABC-аналіз

## Навіщо робити ABC-аналіз?

ABC-аналіз — це **простий і потужний спосіб розставити пріоритети**. Він допомагає відповісти на питання:

> Які товари приносять найбільшу частку виручки і де варто зосередити зусилля?

Ми майже завжди стикаємося з тим, що:

* **20% товарів** приносять **80% виручки** (принцип Парето),
* А більшість товарів створюють **мізерну частку доходу**, але потребують ресурсів (зберігання, логістика, маркетинг).

---

## Застосування

* управління запасами (логістика),
* оптимізація асортименту (retail),
* аналіз виручки (фінанси),
* управління SKU (Stock Keeping Unit/товарна позиція) (продуктова аналітика).

---

## Основна ідея

Поділити товари за **внеском у сукупну виручку**:

| Клас | Виручка | Орієнтовний % товарів | Ціль управління           |
| ---- | ------- | --------------------- | ------------------------- |
| A    | 70–80%  | 10–20%                | Максимальний контроль     |
| B    | 15–25%  | \~30%                 | Пошук покращень           |
| C    | ≤ 5%    | 50–60%                | Автоматизація, скорочення |

---

## Етапи ABC-аналізу

1. **Підготовка даних**: виручка по кожному товару (або SKU).
2. **Агрегація** (якщо потрібно): групування за ID, сума виручки.
3. **Сортування** за спаданням виручки.
4. **Накопичення** частки від загального обсягу.
5. **Класифікація** за порогами: A / B / C.
6. **Інтерпретація та дії**.


## Приклад

In [ ]:
import pandas as pd

data = {
    'item_id': ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A7'],
    'total_sales_value': [1000, 800, 500, 200, 150, 100, 50, 2000]
}

df = pd.DataFrame(data)
df

,item_id,total_sales_value
0,A1,1000
1,A2,800
2,A3,500
3,A4,200
4,A5,150
5,A6,100
6,A7,50
7,A7,2000


In [ ]:
# Крок 1. Групування за товарами (якщо дублюються)
df_grouped = df.groupby('item_id').agg({
    'total_sales_value': 'sum'
}).reset_index()
df_grouped

,item_id,total_sales_value
0,A1,1000
1,A2,800
2,A3,500
3,A4,200
4,A5,150
5,A6,100
6,A7,2050


In [ ]:
# Крок 2. Сортування за спаданням
df_sorted = df_grouped.sort_values('total_sales_value', ascending=False).reset_index()
df_sorted

,index,item_id,total_sales_value
0,6,A7,2050
1,0,A1,1000
2,1,A2,800
3,2,A3,500
4,3,A4,200
5,4,A5,150
6,5,A6,100


In [ ]:
df_sorted['total_sales_value'].cumsum()

,total_sales_value
0,2050
1,3050
2,3850
3,4350
4,4550
5,4700
6,4800


In [ ]:
# Крок 3. Накопичення та відсоток

df_sorted['cum_total'] = df_sorted['total_sales_value'].cumsum()
df_sorted

,index,item_id,total_sales_value,cum_total
0,6,A7,2050,2050
1,0,A1,1000,3050
2,1,A2,800,3850
3,2,A3,500,4350
4,3,A4,200,4550
5,4,A5,150,4700
6,5,A6,100,4800


In [ ]:
df_sorted['total_sales_value'].cumsum()

,total_sales_value
0,2050
1,3050
2,3850
3,4350
4,4550
5,4700
6,4800


In [ ]:
total = df_sorted.total_sales_value.sum()
df_sorted['cum_total_percents'] = df_sorted.cum_total / total
df_sorted

,index,item_id,total_sales_value,cum_total,cum_total_percents
0,6,A7,2050,2050,0.427083
1,0,A1,1000,3050,0.635417
2,1,A2,800,3850,0.802083
3,2,A3,500,4350,0.906250
4,3,A4,200,4550,0.947917
5,4,A5,150,4700,0.979167
6,5,A6,100,4800,1.000000


In [ ]:
# Крок 4. Присвоєння класу
def abc_classify(item):
  if item <= 0.8:
    return 'A'
  if item <= 0.95:
    return 'B'
  return 'C'

df_sorted['ABC'] = df_sorted.cum_total_percents.apply(abc_classify)
df_sorted

,index,item_id,total_sales_value,cum_total,cum_total_percents,ABC
0,6,A7,2050,2050,0.427083,A
1,0,A1,1000,3050,0.635417,A
2,1,A2,800,3850,0.802083,B
3,2,A3,500,4350,0.906250,B
4,3,A4,200,4550,0.947917,B
5,4,A5,150,4700,0.979167,C
6,5,A6,100,4800,1.000000,C


## Що робити після ABC-класифікації?

### Клас A (ключові)

* Максимум уваги: постачання, склад, просування.
* Стежити за залишками, рекламою, знижками.
* Часто аналізувати динаміку.

### Клас B (середня значущість)

* Оптимізувати: умови закупівлі, постачальників.
* Є потенціал зростання — варто подумати про маркетинг.

### Клас C (низькопріоритетні)

* Мінімізувати запаси.
* Об’єднувати SKU, видаляти неефективні.
* Можливо — перевести на автоматичний контроль.


# 🔷 XYZ-аналіз

## Навіщо робити XYZ-аналіз?

Якщо ABC-аналіз показує, **що важливо за виручкою**, то XYZ-аналіз показує:

> Наскільки **стабільний попит** на товар?

Ви можете мати товар із високою виручкою, але якщо він продається стрибкоподібно і нестабільно — з ним треба поводитися обережно (ризик залишків, дефіциту, списань).

---

## Основна ідея

Оцінити **прогнозованість попиту** за коефіцієнтом варіації.
*Коефіцієнт варіації* — це відносна міра розкиду даних, яка показує, наскільки сильно варіюються значення відносно їхнього середнього.

| Клас | Стабільність попиту     | Інтерпретація                      |
| ---- | ----------------------- | ---------------------------------- |
| X    | Висока стабільність     | Попит майже не змінюється          |
| Y    | Середня прогнозованість | Є коливання, але в межах норми     |
| Z    | Низька стабільність     | Хаотичний, сезонний або випадковий |

---

## Як вимірюється?

Через **коефіцієнт варіації (CV)**:

$$
\text{CV} = \frac{\text{Стандартне відхилення}}{\text{Середнє значення}}
$$

---

## Етапи XYZ-аналізу

1. Підготовка даних: продажі за періодами (наприклад, за місяцями).
2. Розрахунок середнього та стандартного відхилення по кожному товару.
3. Розрахунок коефіцієнта варіації (CV).
4. Класифікація на X/Y/Z.
5. Інтерпретація та дії.


## Приклад

In [ ]:
data = {
    'item_id': ['A1', 'A2', 'A3', 'A4', 'A5'],
    'jan': [100, 120, 90, 200, 10],
    'feb': [100, 80, 100, 250, 30],
    'mar': [100, 110, 105, 50, 90],
    'apr': [100, 95, 95, 300, 70]
}

df = pd.DataFrame(data)
df

,item_id,jan,feb,mar,apr
0,A1,100,100,100,100
1,A2,120,80,110,95
2,A3,90,100,105,95
3,A4,200,250,50,300
4,A5,10,30,90,70


In [ ]:
# Крок 1. Розрахунок середнього та стандартного відхилення
df['mean_sales'] = df[['jan', 'feb', 'mar', 'apr']].mean(axis=1)
df['std_sales'] = df[['jan', 'feb', 'mar', 'apr']].std(axis=1)

# Розраховуємо середнє та стандартне відхилення за місяцями


df

,item_id,jan,feb,mar,apr,mean_sales,std_sales
0,A1,100,100,100,100,100.00,0.000000
1,A2,120,80,110,95,101.25,17.500000
2,A3,90,100,105,95,97.50,6.454972
3,A4,200,250,50,300,200.00,108.012345
4,A5,10,30,90,70,50.00,36.514837


In [ ]:
# Крок 2. Коефіцієнт варіації
df['cv'] = (df.std_sales / df.mean_sales)
df

,item_id,jan,feb,mar,apr,mean_sales,std_sales,cv
0,A1,100,100,100,100,100.00,0.000000,0.000000
1,A2,120,80,110,95,101.25,17.500000,0.172840
2,A3,90,100,105,95,97.50,6.454972,0.066205
3,A4,200,250,50,300,200.00,108.012345,0.540062
4,A5,10,30,90,70,50.00,36.514837,0.730297


In [ ]:
# Крок 3. Класифікація за порогами
def xyz_classify(item):
  if item <= 0.25:
    return 'X'
  if item <= 0.50:
    return 'Y'
  return 'Z'


df['xyz'] = df.cv.apply(xyz_classify)
df

,item_id,jan,feb,mar,apr,mean_sales,std_sales,cv,xyz
0,A1,100,100,100,100,100.00,0.000000,0.000000,X
1,A2,120,80,110,95,101.25,17.500000,0.172840,X
2,A3,90,100,105,95,97.50,6.454972,0.066205,X
3,A4,200,250,50,300,200.00,108.012345,0.540062,Z
4,A5,10,30,90,70,50.00,36.514837,0.730297,Z


## Як інтерпретувати?

| Клас | Що це означає                              | Що робити?                                          |
| ---- | ------------------------------------------ | --------------------------------------------------- |
| X    | Попит стабільний, легко прогнозувати       | Планувати точно, тримати у постійній наявності      |
| Y    | Середня прогнозованість, помірні коливання | Підлаштовуватися під тренди, враховувати сезонність |
| Z    | Нестабільний, хаотичний попит              | Мінімальні запаси, уникати автоматичного поповнення |


# 🔷 ABC+XYZ матриця

## Комбінація ABC і XYZ: матриця 3×3

Об’єднуючи обидва аналізи, отримуємо **матрицю з 9 категорій товарів**:

|       | **X (стабільний)** | **Y (помірний)** | **Z (хаотичний)** |
| ----- | ------------------ | ---------------- | ----------------- |
| **A** | AX                 | AY               | AZ                |
| **B** | BX                 | BY               | BZ                |
| **C** | CX                 | CY               | CZ                |

---

## Інтерпретація комірок ABC+XYZ

### 1. **AX — стратегічні товари**

* Висока цінність і стабільний попит.
* Постійна наявність на складі.
* Пріоритетне управління.

---

### 2. **AY — важливі, але з коливаннями**

* Дає великий оборот, але попит нестабільний.
* Потрібен регулярний аналіз попиту та гнучкі закупівлі.

---

### 3. **AZ — важливі, але непередбачувані**

* Високий дохід, але попит хаотичний.
* Ризики переваги/дефіциту.
* Краще закуповувати по факту, за замовленнями.

---

### 4. **BX — середня цінність, стабільний попит**

* Добре прогнозуються.
* Можна закуповувати за планом.
* Рівень контролю трохи нижчий, ніж у AX.

---

### 5. **BY — помірні за значущістю і стабільністю**

* Стандартна стратегія управління запасами.
* Моніторинг, але не пріоритет.

---

### 6. **BZ — середня важливість, хаотичний попит**

* Бажано скоротити склад або перейти на замовлення під клієнта.

---

### 7. **CX — дешеві, але стабільні**

* Можна закуповувати великими партіями.
* Підходять для автоматичного поповнення.

---

### 8. **CY — дешеві, помірно нестабільні**

* Можливо мати запас, але обережно.
* Баланс між витратами та ризиками.

---

### 9. **CZ — дешеві і хаотичні**

* Кандидати на виведення з асортименту.
* Закупівля по факту або виключення.

---

## Як використовувати на практиці?

* **AX, BX** — автоматизація закупівель, регулярний перегляд.
* **AZ, BZ, CZ** — зменшення складських залишків, закупівля по замовленню.
* **CY, CZ** — кандидати на видалення/заміну.
* **ABC** — формує пріоритет за важливістю.
* **XYZ** — формує пріоритет за прогнозованістю.


# 🔷 Приклад

### ✅ A/X — **Золотий фонд**

* Високий внесок у виручку
* Стабільний попит
* **Що робити:**

  * Тримати максимальний пріоритет
  * Гарантувати постійну доступність
  * Ретельно відстежувати продажі
  * Не допускати out-of-stock за жодних умов
  * Застосовувати автоматичне поповнення

---

### ✅ A/Y — **Важливі, але коливаються**

* Висока цінність
* Середня стабільність
* **Що робити:**

  * Відстежувати тренди та сезонність
  * Використовувати просунутий прогноз попиту (наприклад, ковзні середні)
  * Тримати середній рівень запасів
  * Планувати промо, якщо спадання повторюється по часу

---

### ✅ A/Z — **Ризикові, але дорогі**

* Дуже важливі
* Дуже нестабільні
* **Що робити:**

  * Перевірити причини нестабільності: акції, збої постачання, сезонність
  * Застосувати підхід JIT (just in time)
  * Уникати переваги запасів
  * Узгодити зі стратегічним відділом: «тримати під замовлення» або «пульсова закупівля»

---

### 🟨 B/X — **Надійні робочі конячки**

* Середня цінність
* Надійний попит
* **Що робити:**

  * Підтримувати запаси стабільно
  * Можна автоматизувати закупівлі
  * Контролювати залишки, але без надмірностей

---

### 🟨 B/Y — **Менш стабільні, середня цінність**

* Складніше прогнозувати
* **Що робити:**

  * Використовувати помірне буферне зберігання
  * Відстежувати сплески
  * Застосовувати ручний контроль або ML-модель на основі коливань

---

### 🟨 B/Z — **Нестабільні та середньоцінні**

* Погано прогнозуються
* **Що робити:**

  * Мінімізувати запаси
  * Під замовлення, по заявці
  * Не тримати на складі «про запас»

---

### 🟥 C/X — **Багато, стабільно, але дешево**

* Мало приносять, але добре прогнозуються
* **Що робити:**

  * Використовувати автоматичне поповнення за залишковим принципом
  * Продавати через long-tail канали (маркетплейси, акції, розпродажі)
  * Можна формувати промо-набори

---

### 🟥 C/Y — **Багато, середньо передбачувано**

* Неефективні та нестабільні
* **Що робити:**

  * Провести аналіз — навіщо вони потрібні?
  * Скоротити номенклатуру
  * Передати на знижену ціну або розпродаж

---

### 🟥 C/Z — **Нижній баласт**

* Мінімальний внесок + хаотичний попит
* **Що робити:**

  * Оптимізувати: прибрати з зберігання
  * Списати, ліквідувати
  * Перевести на дозаказ за запитом клієнта
  * Не тримати на складі
